In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [52]:
import openai

client = openai.OpenAI()
aclient = openai.AsyncOpenAI()

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello, how are you?"},
]

response = await aclient.chat.completions.create(
    model="gpt-5.4-nano",
    messages=messages,
)
response


ChatCompletion(id='chatcmpl-Dwo6fzBtfk4IiQL4R4SepPECDDadc', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! I’m doing well—thanks for asking. How are you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1782908005, model='gpt-5.4-nano-2026-03-17', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=19, prompt_tokens=22, total_tokens=41, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [ ]:
import dspy
class MyCustomLLM(dspy.BaseLM):
    forward_contract = "legacy"
    def __init__(self, model: str, **kwargs):
        super().__init__(model=model, **kwargs)
        self.provider, self.model_id = model.split("/")
        # gpt-5.4 series has changed the max_tokens parameter to max_completion_tokens
        kwargs['max_completion_tokens'] = kwargs.pop('max_tokens', 1000)
        self.kwargs = kwargs

    def forward(self,
        prompt: str | None = None,
        messages: list[dict] | None = None,
        **kwargs) -> str:
        messages = messages or [{"role": "user", "content": prompt}]

        # use the OpenAI client to show an example
        # your custom LLM backend logic would go here

        client = openai.OpenAI()
        response = client.chat.completions.create(
            model=self.model_id,
            messages=messages,
            **self.kwargs, **kwargs
        )
        return response

    async def aforward(self,
        prompt: str | None = None,
        messages: list[dict] | None = None,
        **kwargs) -> str:
        messages = messages or [{"role": "user", "content": prompt}]

        # use the AsyncOpenAI client to show an example
        # your custom LLM backend logic would go here
        aclient = openai.AsyncOpenAI()
        response = await aclient.chat.completions.create(
            model=self.model_id,
            messages=messages,
            **self.kwargs, **kwargs
        )
        return response

In [50]:
my_custom_llm = MyCustomLLM(model="my_llm/gpt-5.4-nano")
predictor = dspy.Predict("q -> a")
with dspy.context(lm = my_custom_llm):
    print(predictor(q="Why did the chicken cross the kitchen?"))

Prediction(
    a='It was trying to get to the other side of the pantry.'
)


In [51]:
my_custom_llm = MyCustomLLM(model="my_llm/gpt-5.4-nano")
with dspy.context(lm = my_custom_llm):
    print(await predictor.aforward(q="Why did the chicken cross the kitchen?"))

Prediction(
    a='To get to the other side.'
)
